# Summed 1.2 GHz + 60 MHz offset-PDH drive

DAC230/0 → broadband RF amplifier → phase EOM. DAC228/2 → PDH demodulator LO.
The two digital outputs have independently adjustable amplitudes; the reference also has adjustable phase.
Use the matching **PDH** bit/HWH. Perform your normal RFSoC clock initialization and DAC-tile startup first.
Loading a bitstream interrupts all existing functions; attach without download only when this matching image is already running.
The old RF-chain FRA and the old DAC228/2 frequency-waveform output are replaced in this build.
The two feedback PLLs, intensity controller and closed-loop FRA remain available.

Import `xrfdc` before constructing `Overlay` in your startup notebook. If `ol` was
created before that import, the next cell reattaches with `download=False` to fix
its RFDC driver binding. Keep the matching bit/HWH on disk; do not rerun clock
initialization or download the bitstream to fix a `DefaultIP`/`dac_tiles` error.

DAC230/0 is on the **JHC3** breakout bank. This new tile runs at **6.88128 GS/s**, using the existing **491.52 MHz reference distributed from DAC229**. No new external clock connection is needed. The former DAC229/2 PDH output is now zero.


In [ ]:
import importlib
from pathlib import Path
import xrfdc  # Register the RFDC driver BEFORE constructing Overlay.
from pynq import Overlay
import pdh_tones as pdh

importlib.invalidate_caches()
pdh = importlib.reload(pdh)
# If ol already exists, keep using it. Otherwise attach to the matching file:
# ol = Overlay('/path/to/PDH_DAC230_1200_60/TransportPhaseLockFF.bit', download=False)
# Repair an ol created before importing xrfdc; no bitstream download or restart.
if not hasattr(ol.usp_rf_data_converter_0, 'dac_tiles'):
    ol = Overlay(ol.bitfile_name, download=False)
rfdc = ol.usp_rf_data_converter_0
print('RFDC driver:', type(rfdc))
assert hasattr(rfdc, 'dac_tiles'), (ol.ip_dict['usp_rf_data_converter_0']['type'],
                                  xrfdc.RFdc.bindto)
print('Helper:', pdh.__file__)
print('Helper version:', getattr(pdh, 'HELPER_VERSION', 'old helper; copy updated pdh_tones.py'))
pdh.check(ol)
print(pdh.status(ol))


## Configure once, with amplitudes zero
This configures **DAC230/0 and DAC228/2** for **R2R coarse mixer bypass**, both with ×2 interpolation. The sum is generated directly in RTL; do not set a 630 MHz or 1.2 GHz RFDC mixer.
The helper starts the new DAC230 tile if it is not ready. Existing tiles are not restarted. Run the normal board clock startup first so DAC229 supplies its distributed reference.
Calibrate the physical reference phase after a converter/bitstream restart. The helper requires PDH feature ID `0x50444802`.


In [ ]:
print(pdh.configure(ol))  # All requested amplitudes default to zero.
print('DAC230 state:', rfdc.IPStatus['DACTileStatus'][2]['TileState'])  # 15 = ready
print('DAC230 PLL:', rfdc.dac_tiles[2].PLLLockStatus)
print('DAC230/0 mixer:', dict(rfdc.dac_tiles[2].blocks[0].MixerSettings))
print('DAC228/2 mixer:', dict(rfdc.dac_tiles[0].blocks[2].MixerSettings))
print('DAC230 PLL settings:', dict(rfdc.dac_tiles[2].PLLConfig))
print('DAC230/0 block status:', dict(rfdc.dac_tiles[2].blocks[0].BlockStatus))


## Check frequencies and levels before optimizing the optical lock
The following are low digital test levels, not calibrated voltages. Check available RF gain and the measured spectrum at the EOM input.
The sum of the carrier and 60 MHz digital amplitudes must be ≤1; reference amplitude has its own limit.
No feedback-controller settings are changed by these calls.

**Expected electrical spectrum:** 1.2 GHz plus 60 MHz. The 1.14/1.26 GHz sidebands are produced in the **optical** field by the EOM, not in this electrical sum.
Start with each tone alone before summing; record frequencies and unwanted-line levels in dBc. The software/RTL checks cannot certify analog RF spectral purity.


In [ ]:
# First: carrier only. Measure the spectrum on DAC230/0.
print(pdh.set_levels(ol, carrier_amplitude=0.10, sideband_amplitude=0,
                     reference_amplitude=0.10))
# Then run separately for a 60 MHz-only check:
# print(pdh.set_levels(ol, carrier_amplitude=0, sideband_amplitude=0.10,
#                      reference_amplitude=0.10))


In [ ]:
carrier_amplitude = 0.10
sideband_amplitude = None  # Default: 0.587654 * carrier_amplitude; 0 disables it.
reference_amplitude = 0.10
reference_phase_deg = 0.0

print(pdh.set_levels(
    ol, carrier_amplitude=carrier_amplitude, sideband_amplitude=sideband_amplitude,
    reference_amplitude=reference_amplitude, reference_phase_deg=reference_phase_deg,
))


## Optimize the offset and PDH modulation depths
For the ideal maximum discriminator slope, target optical phase modulation indices βoffset≈1.84 and βPDH≈1.08 rad.
With β=πVpeak/Vπ, these require **0.586 Vπ peak at 1.2 GHz** and **0.344 Vπ peak at 60 MHz**.
If RF voltage transfer and Vπ were equal, the voltage ratio would be 0.588 (−4.62 dB in RF power).
Digital weights 0.55 and 0.3232 have that ratio, but do not establish the absolute modulation depths: calibrate at the EOM.

1. Set `sideband_amplitude=0`; maximize the selected +1.2 GHz optical sideband by adjusting carrier amplitude / RF gain.
2. Raise `sideband_amplitude` and maximize measured PDH slope while optimizing demodulation phase.
3. Check that amplifier compression, unwanted cavity resonances and residual amplitude modulation do not worsen lock noise.

The reference-output amplitude only sets demodulator LO drive; it is independent of optical modulation depth.
See `docs/pdh_tones.md` for the optical-spectrum derivation and physical assumptions.


## Adjust demodulation phase without restarting oscillators
Positive phase advances the reference cosine. The actual analog phase includes tile, filter and cable offsets.
Use this call for live tuning; do not repeatedly call `configure()` or change the RFDC mixer frequency.
Large changes are coefficient steps, not ramps, and can disturb a lock.


In [ ]:
reference_phase_deg = 35.0
print(pdh.set_reference_phase(ol, reference_phase_deg))


## Inspect state or mute
Any sticky clipping/streaming fault mutes both outputs. Inspect and fix its cause, then use `configure()` to rearm.
The nominal frequencies share the existing clock reference, but absolute analog phase across the DAC tiles is not guaranteed to repeat after a restart.


In [ ]:
print(pdh.status(ol))


In [ ]:
# Run when you want both PDH outputs off; other channels are unaffected.
pdh.stop(ol)
